# Business Analytics EDA for the Ride-Sharing Platform

This notebook loads the exported analytics dataset, explores trip behavior, and derives portfolio-ready business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
data_path = Path('sample_analytics_export.csv')
df = pd.read_csv(data_path)
df['trip_date'] = pd.to_datetime(df['trip_date'])
df['status'] = df['status'].str.upper()
df['is_cancelled'] = df['status'].isin(['CANCELLED', 'CANCELADA', 'CANCELADO'])
df['revenue_cop'] = pd.to_numeric(df['revenue_cop'])
df['duration_minutes'] = pd.to_numeric(df['duration_minutes'])
df['hour'] = pd.to_numeric(df['hour'])
df.head()

In [ ]:
summary = {
    'rows': len(df),
    'cities': df['city'].nunique(),
    'drivers': df['driver'].nunique(),
    'customers': df['customer'].nunique(),
    'total_revenue': df['revenue_cop'].sum(),
    'cancellation_rate': df['is_cancelled'].mean() * 100,
    'avg_duration': df.loc[~df['is_cancelled'], 'duration_minutes'].mean()
}
summary

In [ ]:
daily = df.groupby('trip_date', as_index=False).agg(rides=('trip_id', 'count'), revenue=('revenue_cop', 'sum'))
hourly = df.groupby('hour', as_index=False).agg(rides=('trip_id', 'count'))
city = df.groupby('city', as_index=False).agg(rides=('trip_id', 'count'), revenue=('revenue_cop', 'sum'))
drivers = df.groupby('driver', as_index=False).agg(rides=('trip_id', 'count')).sort_values('rides', ascending=False)
customers = df.groupby('customer', as_index=False).agg(rides=('trip_id', 'count')).sort_values('rides', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.lineplot(data=daily, x='trip_date', y='revenue', marker='o', ax=axes[0, 0], color='#1f77b4')
axes[0, 0].set_title('Revenue per Day')
axes[0, 0].tick_params(axis='x', rotation=45)

sns.barplot(data=hourly, x='hour', y='rides', ax=axes[0, 1], color='#ff7f0e')
axes[0, 1].set_title('Peak Demand Hours')

sns.barplot(data=city, x='city', y='rides', ax=axes[1, 0], palette='Blues_r')
axes[1, 0].set_title('Trips by City')
axes[1, 0].tick_params(axis='x', rotation=30)

sns.barplot(data=drivers.head(5), x='rides', y='driver', ax=axes[1, 1], palette='Greens_r')
axes[1, 1].set_title('Top Drivers')
plt.tight_layout()

In [ ]:
insights = [
    f'Total trips analyzed: {summary["rows"]}.',
    f'Revenue concentrates in the strongest city mix, with {city.sort_values("revenue", ascending=False).iloc[0]["city"]} leading revenue generation.',
    f'Peak demand occurs around hour {hourly.sort_values("rides", ascending=False).iloc[0]["hour"]}.',
    f'Cancellation rate is {summary["cancellation_rate"]:.2f}% and should be monitored weekly.',
    f'Average trip duration for completed rides is {summary["avg_duration"]:.1f} minutes.',
    f'Top driver concentration appears in {drivers.iloc[0]["driver"]} with {drivers.iloc[0]["rides"]} trips.',
    f'Top customer concentration appears in {customers.iloc[0]["customer"]} with {customers.iloc[0]["rides"]} trips.',
    f'Revenue per ride can be used as a pricing health metric: {summary["total_revenue"] / max(summary["rows"], 1):.2f} COP/ride.',
    f'City-level demand shows where fleet allocation should be prioritized.',
    f'This hybrid dataset is suitable for backend, database, and BI storytelling in interviews.'
]
for item in insights:
    print('-', item)